In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, len(files))

In [ ]:
import os

for name in os.listdir('/kaggle/input'):
    path = os.path.join('/kaggle/input', name)
    n_files = sum(len(f) for _, _, f in os.walk(path))
    print(name, '->', n_files, 'files')
    print(sorted(os.listdir(path))[:10])

In [ ]:
import os, glob
import pandas as pd

hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
print('train.csv found at:', hits)

BASE = os.path.dirname(hits[0])
print('BASE =', BASE)
print('items in BASE:', sorted(os.listdir(BASE)))

for f in ['train.csv', 'test.csv', 'sample_submission.csv']:
    df = pd.read_csv(os.path.join(BASE, f))
    print()
    print(f, df.shape)
    print(df.head(3))
    print(df.dtypes)

In [ ]:
for root, dirs, files in os.walk(BASE):
    ext = sorted({os.path.splitext(f)[1] for f in files})
    print(root, '| folders:', dirs[:5], '| files:', len(files), '| extensions:', ext)
    if files:
        print('   sample files:', sorted(files)[:3])

In [ ]:
import sys, shutil
import torch
import psutil

print('python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print('gpu', i, torch.cuda.get_device_name(i),
          round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1), 'GB')
print('RAM GB:', round(psutil.virtual_memory().total / 1e9, 1))
print('free disk (working) GB:', round(shutil.disk_usage('/kaggle/working').free / 1e9, 1))

# internet and HuggingFace Hub check
try:
    from huggingface_hub import hf_hub_download
    p = hf_hub_download('openai/whisper-tiny', 'config.json')
    print('huggingface hub access: OK')
except Exception as e:
    print('huggingface hub access: FAILED ->', repr(e)[:200])

In [ ]:
import importlib, shutil

pkgs = ['numpy', 'pandas', 'scipy', 'sklearn', 'librosa', 'soundfile',
        'transformers', 'lightgbm', 'xgboost', 'catboost',
        'faster_whisper', 'language_tool_python', 'shap',
        'matplotlib', 'seaborn']

for p in pkgs:
    try:
        m = importlib.import_module(p)
        print(f'{p:25s} OK      {getattr(m, "__version__", "")}')
    except Exception:
        print(f'{p:25s} MISSING')

print('java path:', shutil.which('java'))

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = secret_value_0

print("HF_TOKEN successfully loaded!")


In [ ]:
!pip install -q faster-whisper language-tool-python

import subprocess
print(subprocess.run(['java', '-version'], capture_output=True, text=True).stderr)

import faster_whisper, language_tool_python
print('faster_whisper imported OK')

tool = language_tool_python.LanguageTool('en-US')
matches = tool.check('He go to school yesterday and they was happy.')
print('grammar issues found:', len(matches))
if matches:
    print(matches[0])
tool.close()

### PHASE 1

In [ ]:
import os, glob, hashlib, warnings
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

SEED = 42
BASE = '/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final'
TRAIN_DIR = f'{BASE}/train'
TEST_DIR = f'{BASE}/test'
WORK = '/kaggle/working'

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')
sub = pd.read_csv(f'{BASE}/sample_submission.csv')
print(train.shape, test.shape, sub.shape)

In [ ]:
train_files = set(os.listdir(TRAIN_DIR))
test_files = set(os.listdir(TEST_DIR))

print('train.csv rows not in train folder :', len(set(train.filename) - train_files))
print('train folder files not in train.csv:', len(train_files - set(train.filename)))
print('test.csv rows not in test folder   :', len(set(test.filename) - test_files))
print('test folder files not in test.csv  :', len(test_files - set(test.filename)))
print()
print('sample_submission files not in test.csv:', len(set(sub.filename) - set(test.filename)))
print('test.csv files not in sample_submission:', len(set(test.filename) - set(sub.filename)))
print('sample_submission files missing in test folder:', len(set(sub.filename) - test_files))
print()
print('duplicate filenames  train/test/sub:',
      train.filename.duplicated().sum(), test.filename.duplicated().sum(), sub.filename.duplicated().sum())
print('null labels in train:', train.label.isna().sum())
print('files excluded from submission:', sorted(set(test.filename) - set(sub.filename)))

In [ ]:
print(train.label.describe())
print()
print(train.label.value_counts().sort_index())

dummy_rmse = np.sqrt(np.mean((train.label - train.label.mean()) ** 2))
print()
print('RMSE of always predicting the mean (dummy baseline):', round(dummy_rmse, 4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train.label, bins=20, kde=False, ax=ax[0])
ax[0].set_title('Label distribution (train)')
ax[0].set_xlabel('Grammar score')
vc = train.label.value_counts().sort_index()
ax[1].bar(vc.index.astype(str), vc.values)
ax[1].set_title('Count per distinct label')
ax[1].set_xlabel('Grammar score')
plt.tight_layout()
plt.show()

In [ ]:
def audio_info(path):
    i = sf.info(path)
    return dict(sr=i.samplerate, channels=i.channels, frames=i.frames,
                duration=i.frames / i.samplerate, subtype=i.subtype)

rows = []
for split, df, d in [('train', train, TRAIN_DIR), ('test', test, TEST_DIR)]:
    for fn in df.filename:
        r = audio_info(os.path.join(d, fn))
        r.update(filename=fn, split=split)
        rows.append(r)
meta = pd.DataFrame(rows)
meta = meta.merge(train[['filename', 'label']], on='filename', how='left')

print(meta.groupby('split')[['duration']].describe().T)
print()
print('sample rates :', meta.groupby('split').sr.value_counts().to_dict())
print('channels     :', meta.groupby('split').channels.value_counts().to_dict())
print('subtypes     :', meta.subtype.value_counts().to_dict())

tr = meta[meta.split == 'train']
rho, p = spearmanr(tr.duration, tr.label)
print('\nSpearman(duration, label):', round(rho, 3), 'p =', round(p, 4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=meta, x='duration', hue='split', bins=30, ax=ax[0], element='step')
ax[0].set_title('Audio duration (seconds)')
sns.scatterplot(data=tr, x='duration', y='label', alpha=0.4, ax=ax[1])
ax[1].set_title('Duration vs label')
plt.tight_layout()
plt.show()

In [ ]:
import librosa
from joblib import Parallel, delayed

def signal_stats(path):
    y, sr = librosa.load(path, sr=16000, mono=True)
    rms = float(np.sqrt(np.mean(y ** 2)) + 1e-12)
    intervals = librosa.effects.split(y, top_db=30)
    speech_ratio = float(sum(e - s for s, e in intervals) / max(len(y), 1))
    with open(path, 'rb') as f:
        md5 = hashlib.md5(f.read()).hexdigest()
    return dict(rms_db=20 * np.log10(rms), peak=float(np.max(np.abs(y))),
                clip_ratio=float(np.mean(np.abs(y) > 0.99)),
                speech_ratio=speech_ratio, md5=md5)

paths = [os.path.join(TRAIN_DIR if s == 'train' else TEST_DIR, fn)
         for fn, s in zip(meta.filename, meta.split)]
stats = Parallel(n_jobs=4)(delayed(signal_stats)(p) for p in paths)
meta = pd.concat([meta.reset_index(drop=True), pd.DataFrame(stats)], axis=1)
meta.to_csv(f'{WORK}/audio_meta.csv', index=False)

tr = meta[meta.split == 'train']
print('Spearman with label:')
for c in ['duration', 'rms_db', 'peak', 'clip_ratio', 'speech_ratio']:
    rho, p = spearmanr(tr[c], tr.label)
    print(f'  {c:14s} rho={rho:+.3f}  p={p:.4f}')

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, c in zip(ax, ['rms_db', 'speech_ratio', 'peak']):
    sns.scatterplot(data=tr, x=c, y='label', alpha=0.4, ax=a)
    a.set_title(f'{c} vs label')
plt.tight_layout()
plt.show()

# duplicate audio check
g = meta.groupby('md5').filename.apply(list)
dups = g[g.apply(len) > 1]
print('\nduplicate audio groups:', len(dups))
for files in dups.head(10):
    lab = meta[meta.filename.isin(files)][['filename', 'split', 'label']]
    print(lab.to_string(index=False), '\n')

In [ ]:
lo = tr.sort_values('label').iloc[0]
hi = tr.sort_values('label').iloc[-1]

fig, ax = plt.subplots(2, 2, figsize=(14, 7))
for j, row in enumerate([lo, hi]):
    y, sr = librosa.load(os.path.join(TRAIN_DIR, row.filename), sr=16000)
    t = np.arange(len(y)) / sr
    ax[0, j].plot(t, y, linewidth=0.4)
    ax[0, j].set_title(f'{row.filename} | label={row.label}')
    ax[0, j].set_xlabel('seconds')
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80), ref=np.max)
    librosa.display.specshow(S, sr=sr, x_axis='time', y_axis='mel', ax=ax[1, j])
    ax[1, j].set_title('Mel spectrogram')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

# stratify on 5 equal-sized label bins (robust even if some labels are rare)
strat = pd.qcut(train.label.rank(method='first'), 5, labels=False)

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
folds = pd.DataFrame({'filename': train.filename})
for k, (_, val_idx) in enumerate(rskf.split(train, strat)):
    rep, fold = divmod(k, 5)
    folds.loc[val_idx, f'rep{rep}'] = fold
folds[['rep0', 'rep1', 'rep2']] = folds[['rep0', 'rep1', 'rep2']].astype(int)
folds.to_csv(f'{WORK}/folds.csv', index=False)

chk = train.assign(fold=folds.rep0)
print(chk.groupby('fold').label.agg(['count', 'mean', 'std']).round(3))

def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

# honest dummy baseline under our CV: predict the training-fold mean
oof = np.zeros(len(train))
for f in range(5):
    va = folds.rep0 == f
    oof[va] = train.label[~va].mean()
print('\nDummy CV RMSE (rep0):', round(rmse(train.label, oof), 4))

In [ ]:
# label 0 audio signature
from sklearn.metrics import roc_auc_score
from joblib import Parallel, delayed

def noise_feats(path):
    y, sr = librosa.load(path, sr=16000, mono=True)
    rms = librosa.feature.rms(y=y, frame_length=1024, hop_length=256)[0] + 1e-9
    db = 20 * np.log10(rms)
    flat = librosa.feature.spectral_flatness(y=y, n_fft=1024, hop_length=256)[0]
    return dict(db_p10=float(np.percentile(db, 10)),
                db_p50=float(np.percentile(db, 50)),
                db_p90=float(np.percentile(db, 90)),
                dyn_range=float(np.percentile(db, 90) - np.percentile(db, 10)),
                flat_mean=float(flat.mean()),
                flat_p90=float(np.percentile(flat, 90)))

paths = [os.path.join(TRAIN_DIR if s == 'train' else TEST_DIR, fn)
         for fn, s in zip(meta.filename, meta.split)]
nf = Parallel(n_jobs=4)(delayed(noise_feats)(p) for p in paths)
meta = pd.concat([meta.reset_index(drop=True), pd.DataFrame(nf)], axis=1)
meta.to_csv(f'{WORK}/audio_meta.csv', index=False)

tr = meta[meta.split == 'train'].copy()
tr['is_zero'] = (tr.label == 0).astype(int)
cols = ['duration', 'rms_db', 'peak', 'speech_ratio',
        'db_p10', 'db_p50', 'db_p90', 'dyn_range', 'flat_mean', 'flat_p90']

print(tr.groupby('is_zero')[cols].agg(['mean', 'std']).round(3).T)
print()
print('AUC of each feature for detecting label==0 (0.5 = useless, 1.0 or 0.0 = perfect):')
for c in cols:
    print(f'  {c:12s} {roc_auc_score(tr.is_zero, tr[c]):.3f}')

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, c in zip(ax, ['db_p10', 'dyn_range', 'flat_mean', 'rms_db']):
    sns.boxplot(data=tr, x='is_zero', y=c, ax=a)
    a.set_title(c)
plt.tight_layout()
plt.show()

print('\ntest set: how many files look like the zero cluster? (db_p10 above the zero-group median)')
thr = tr[tr.is_zero == 1].db_p10.median()
te = meta[meta.split == 'test']
print('threshold db_p10:', round(thr, 2), '| test files above it:', int((te.db_p10 > thr).sum()), 'of', len(te))
print('train files above it:', int((tr.db_p10 > thr).sum()), '(zeros among them:', int(tr[tr.db_p10 > thr].is_zero.sum()), ')')

In [ ]:
def dur_group(d):
    if d < 40:
        return 'short_lt40'
    if d < 52:
        return 'dur_45s'
    return 'dur_60s'

meta['dur_group'] = meta.duration.apply(dur_group)
print(pd.crosstab(meta.split, meta.dur_group))
print()
print(pd.crosstab(meta.split, meta.dur_group, normalize='index').round(3))

tr = meta[meta.split == 'train'].copy()
print()
print(tr.groupby('dur_group').label.agg(['count', 'mean', 'std']).round(3))
print()
print('share of label==0 per duration group:')
print((tr.label == 0).groupby(tr.dur_group).mean().round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=tr, x='label', hue='dur_group', ax=ax[0])
ax[0].set_title('Label counts by duration group (train)')
sns.boxplot(data=tr, x='dur_group', y='label', ax=ax[1])
ax[1].set_title('Label by duration group (train)')
plt.tight_layout()
plt.show()

In [ ]:
train_files = set(os.listdir(TRAIN_DIR))
test_files = set(os.listdir(TEST_DIR))
sub_ids = set(sub.filename)

print('sample_submission files found in test folder :', len(sub_ids & test_files))
print('sample_submission files found in train folder:', len(sub_ids & train_files))
print('sample_submission files found in neither     :', len(sub_ids - test_files - train_files))

num = lambda s: int(s.split('_')[1].split('.')[0])
for name, ids in [('train', train.filename), ('test', test.filename), ('sample_sub', sub.filename)]:
    n = ids.map(num)
    print(f'{name:11s} id range: {n.min()} to {n.max()}  count: {len(n)}')

print('\nsample of overlap with test:', sorted(sub_ids & test_files)[:15])
print('sample of ids in neither  :', sorted(sub_ids - test_files - train_files)[:15])

In [ ]:
common = sorted(set(train.filename) & set(test.filename))
print('filenames present in BOTH train and test folders:', len(common), '| examples:', common[:4])

meta['uid'] = meta.split + '/' + meta.filename
print('unique uid:', meta.uid.nunique(), 'of', len(meta))

pair = meta[meta.filename.isin(common)].pivot(index='filename', columns='split', values='md5')
print('same filename AND identical audio:', int((pair['train'] == pair['test']).sum()), 'of', len(pair))

# fix: earlier merge copied train labels onto test rows with the same name
meta.loc[meta.split == 'test', 'label'] = np.nan

zero_rule = lambda d: (d.flat_mean > 0.46) & (d.dyn_range < 6)
tr_m = meta[meta.split == 'train']
te_m = meta[meta.split == 'test']
fired = zero_rule(tr_m)
print('TRAIN: rule fires on', int(fired.sum()),
      '| true zeros caught:', int((tr_m[fired].label == 0).sum()),
      'of', int((tr_m.label == 0).sum()))
print('TEST : rule fires on', int(zero_rule(te_m).sum()), 'of', len(te_m))
print(te_m.sort_values('flat_mean', ascending=False)
      [['uid', 'flat_mean', 'dyn_range', 'db_p10', 'duration']].head(6).to_string(index=False))

meta.to_csv(f'{WORK}/audio_meta.csv', index=False)

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

tr = meta[meta.split == 'train'].reset_index(drop=True)[
    ['uid', 'filename', 'label', 'duration', 'dur_group']].copy()
tr['is_zero'] = (tr.label == 0).astype(int)
tr['dg2'] = np.where(tr.dur_group == 'dur_60s', 'g60', 'g45')

bucket = pd.cut(tr.label, [-0.1, 0.1, 2.25, 3.25, 4.25, 5.1], labels=False)
strat = np.where(tr.is_zero == 1, 'zero', bucket.astype(str) + '_' + tr.dg2)
print(pd.Series(strat).value_counts().sort_index())

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
for k, (_, va) in enumerate(rskf.split(tr, strat)):
    rep, f = divmod(k, 5)
    tr.loc[va, f'rep{rep}'] = f
for r in range(3):
    tr[f'rep{r}'] = tr[f'rep{r}'].astype(int)

test_mix = meta[meta.split == 'test'].dur_group.value_counts(normalize=True)
train_mix = tr.dur_group.value_counts(normalize=True)
print('\nimportance weights (test share / train share):')
print((test_mix / train_mix).round(3))
tr['w_test'] = tr.dur_group.map(test_mix / train_mix)

def rmse(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))

def pearson(y, p):
    return float(np.corrcoef(y, p)[0, 1])

def wrmse(y, p, w):
    return float(np.sqrt(np.sum(w * (y - p) ** 2) / np.sum(w)))

def wpearson(y, p, w):
    my, mp = np.average(y, weights=w), np.average(p, weights=w)
    cov = np.sum(w * (y - my) * (p - mp))
    return float(cov / np.sqrt(np.sum(w * (y - my) ** 2) * np.sum(w * (p - mp) ** 2)))

def cv_report(oof):
    y = tr.label.values
    w = tr.w_test.values
    nz = (tr.is_zero == 0).values
    rows = {'all_train': (rmse(y, oof), pearson(y, oof)),
            'non_zero': (rmse(y[nz], oof[nz]), pearson(y[nz], oof[nz]))}
    for g in ['dur_45s', 'dur_60s']:
        m = nz & (tr.dur_group == g).values
        rows[f'non_zero_{g}'] = (rmse(y[m], oof[m]), pearson(y[m], oof[m]))
    rows['non_zero_test_weighted'] = (wrmse(y[nz], oof[nz], w[nz]),
                                      wpearson(y[nz], oof[nz], w[nz]))
    return pd.DataFrame(rows, index=['RMSE', 'Pearson']).T.round(4)

# reference floor: predict the training-fold mean
oof = np.zeros(len(tr))
for f in range(5):
    va = (tr.rep0 == f).values
    oof[va] = tr.label[~va].mean()
print('\nDummy baseline under the new report:')
print(cv_report(oof))

tr.to_csv(f'{WORK}/folds.csv', index=False)

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

feat_cols = ['rms_db', 'peak', 'speech_ratio', 'db_p10', 'db_p50', 'db_p90',
             'dyn_range', 'flat_mean', 'flat_p90']
d = meta[~((meta.split == 'train') & (meta.label == 0))]
X = d[feat_cols].values
y = (d.split == 'test').astype(int).values

clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=8,
                         verbose=-1, random_state=SEED)
p = cross_val_predict(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                      method='predict_proba')[:, 1]
print('adversarial AUC, duration excluded (0.5 = indistinguishable):', round(roc_auc_score(y, p), 3))
clf.fit(X, y)
print(pd.Series(clf.feature_importances_, index=feat_cols).sort_values(ascending=False))

In [ ]:
from IPython.display import Audio, display

for lab in [0.0, 2.0, 5.0]:
    uid = tr[tr.label == lab].uid.iloc[0]
    y_, sr_ = librosa.load(f'{BASE}/{uid}', sr=16000, duration=20)
    print(uid, '| label', lab)
    display(Audio(y_, rate=sr_))

In [ ]:
sub_check = pd.DataFrame({
    'filename': test.filename,
    'label': train.label[train.label > 0].mean() + np.arange(len(test)) * 1e-6})
sub_check.to_csv(f'{WORK}/submission_format_check.csv', index=False)
print(sub_check.shape)

In [ ]:
for g in ['dur_45s', 'dur_60s']:
    d = meta[(meta.dur_group == g) & ~((meta.split == 'train') & (meta.label == 0))]
    X = d[feat_cols].values
    y = (d.split == 'test').astype(int).values
    clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=8,
                             verbose=-1, random_state=SEED)
    p = cross_val_predict(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                          method='predict_proba')[:, 1]
    print(g, '| n_train:', int((y == 0).sum()), '| n_test:', int((y == 1).sum()),
          '| adversarial AUC:', round(roc_auc_score(y, p), 3))